# 3 — Inference: the released RF-DETR baseline

Runs the released RF-DETR checkpoint(s) over the *identical* Brackish val images and writes a COCO results json, so `4_evaluate` can score both models with one function.

> Run **`1_reformat.ipynb` first** (it writes the CFD manifest to Drive). The Brackish frames live on the VM disk at `/content/data/brackish`; if this notebook lands on a fresh runtime, cell 1b re-creates them. Everything else (weights, run dirs, results) is on Drive and persists. On an A100 the package picks bfloat16 automatically; on a T4 it picks fp16 + GradScaler.

## 1 · Drive, paths, code

Mounts Drive, fixes the four paths every cell below uses, clones the branch and installs it editable. Safe to re-run: the clone is wiped and redone each time.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, time, shutil, subprocess, pathlib
DRIVE = '/content/drive/MyDrive/frozen-trunk-detection'
for sub in ('weights', 'runs', 'results', 'results/manifest', 'results/viz'):
    os.makedirs(f'{DRIVE}/{sub}', exist_ok=True)
REPO = '/content/crop-counter'
DATA = '/content/data/brackish'
CFD  = '/content/cfd'
os.makedirs(CFD, exist_ok=True)
print(os.listdir(DRIVE))

%cd /content
!rm -rf crop-counter
!git clone --branch poc/detection-head --depth 1 https://github.com/InsightML/crop-counter.git
%cd /content/crop-counter
!git log --oneline -3
# torch/torchvision come with the Colab image. torchmetrics + termcolor are needed only so
# Meta's dinov3 hubconf imports (it pulls the segmentors); pycocotools for COCO AP; ijson to
# stream the 1.9M-record CFD metadata without json.load-ing it.
!pip install -q -e ".[detection]" ijson
# A running kernel does not re-read site-packages' .pth files, so the editable install is invisible
# to THIS process until restart (subprocess calls like `!python -m cropcounter.train` see it fine).
import sys, importlib
if '/content/crop-counter/src' not in sys.path:
    sys.path.insert(0, '/content/crop-counter/src')
importlib.invalidate_caches()
import cropcounter, torch
print('cropcounter', cropcounter.__file__, '| torch', torch.__version__, '| cuda', torch.cuda.is_available())

## 1b · Make sure the Brackish slice is on this VM

Colab gives each notebook its own runtime, so the frames fetched by `1_reformat.ipynb` are not here unless you attached this notebook to that same session. This cell re-creates the slice only if it is missing (metadata 47 MB, ~14.7k frames from the LILA GCS mirror; ~4–5 min).

In [ ]:
# Idempotent: skip if 1_reformat already populated this runtime.
have = os.path.exists(f'{DATA}/val/annotations.json') and os.path.isdir(f'{DATA}/val/images') and len(os.listdir(f'{DATA}/val/images')) > 0
if not have:
    META = f'{CFD}/community_fish_detection_dataset.json.zip'
    if not os.path.exists(META):
        !wget -q -O {META} https://lilawildlife.blob.core.windows.net/lila-wildlife/community-fish-detection-dataset/community_fish_detection_dataset.json.zip
    !python -m cropcounter.cfd subset --metadata {META} --out {DATA} --sources brackish_dataset --train-cap 100000 --val-cap 100000 --seed 0 --no-progress
    !python -m cropcounter.cfd fetch --subset {DATA} --max-side 1024 --workers 32 --mirror gcs --no-progress
!ls {DATA}/train/images | wc -l; ls {DATA}/val/images | wc -l

## 2 · Baseline — released RF-DETR-Nano (640) through the *identical* harness on the *identical* val images

Weights from the community-fish-detector GitHub release (Apache inference licence). Scored with our `det_metrics.coco_eval` against the same `val/annotations.json` — the same function scores ours. Threshold 0.001 so the PR curve is complete. Optionally repeat for RF-DETR-Small (1024).

In [ ]:
!pip install -q rfdetr supervision
import urllib.request
BASELINES = {
  'rfdetr_nano_640':  'https://github.com/filippovarini/community-fish-detector/releases/download/2026.07.06-release/cfd-rf-detr-nano-640-2026.02.02.cp-011.20260706-release.pth',
  # 'rfdetr_small_1024': 'https://github.com/filippovarini/community-fish-detector/releases/download/2026.07.06-release/cfd-rf-detr-small-1024-2026.06.06.cp-016.20260706-release.pth',
}
os.makedirs(f'{DRIVE}/weights/baselines', exist_ok=True)
for name, url in BASELINES.items():
    p = f'{DRIVE}/weights/baselines/{name}.pth'
    if not os.path.exists(p):
        urllib.request.urlretrieve(url, p)
    print(name, os.path.getsize(p) / 1e6, 'MB')

In [ ]:
from rfdetr import from_checkpoint
from PIL import Image
from tqdm.auto import tqdm
from cropcounter.det_metrics import coco_eval, write_coco_results

gt = json.load(open(f'{DATA}/val/annotations.json'))
id_by_name = {os.path.basename(im['file_name']): im['id'] for im in gt['images']}
baseline_metrics = {}
for name in BASELINES:
    det_model = from_checkpoint(f'{DRIVE}/weights/baselines/{name}.pth')
    dets, t0 = [], time.time()
    for fname, image_id in tqdm(id_by_name.items(), desc=name):
        img = Image.open(f'{DATA}/val/images/{fname}').convert('RGB')
        d = det_model.predict(img, threshold=0.001)
        for (x1, y1, x2, y2), s in zip(d.xyxy, d.confidence):
            dets.append({'image_id': image_id, 'category_id': 1,
                         'bbox': [float(x1), float(y1), float(x2 - x1), float(y2 - y1)], 'score': float(s)})
    secs = time.time() - t0
    write_coco_results(dets, f'{DRIVE}/results/{name}_predictions.json')
    m = coco_eval(f'{DATA}/val/annotations.json', dets)
    m['seconds_per_image'] = secs / len(id_by_name)
    baseline_metrics[name] = m
    print(name, {k: round(v, 4) for k, v in m.items()})
json.dump(baseline_metrics, open(f'{DRIVE}/results/baseline_metrics.json', 'w'), indent=1)

## 3 · Baseline inference FLOPs (measured, not quoted)

In [ ]:
# GFLOPs for the baseline, measured here because `rfdetr` and its checkpoint live in this
# notebook; 4_evaluate reads the number back out of baseline_metrics.json.
from torch.utils.flop_counter import FlopCounterMode
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
for name in BASELINES:
    try:
        core = from_checkpoint(f'{DRIVE}/weights/baselines/{name}.pth').model.model  # rfdetr wraps a LWDETR nn.Module
        side = int(name.split('_')[-1])
        xr = torch.randn(1, 3, side, side, device=device)
        with torch.no_grad(), FlopCounterMode(display=False) as fc:
            core(xr)
        baseline_metrics[name]['gflops'] = round(fc.get_total_flops() / 1e9, 1)
        print(f"{name} ({side}x{side}): {baseline_metrics[name]['gflops']} GFLOPs")
    except Exception as e:
        print('RF-DETR FLOP count skipped:', repr(e)[:200])
json.dump(baseline_metrics, open(f'{DRIVE}/results/baseline_metrics.json', 'w'), indent=1)

## 4 · Next

`4_evaluate.ipynb` — one table, ours vs baseline, same images, same scorer.